In [0]:
from pyspark.sql.functions import * 
from delta.tables import *

In [0]:
%run /Workspace/Users/narutosha289@gmail.com/Consolidated_pipeline/Setup/Utilities 

In [0]:
print(bronze_schema, silver_schema, gold_schema)

Bronze Silver Gold


In [0]:
dbutils.widgets.text("Catalog","FMCG","Catalog") 
dbutils.widgets.text("Data_Source","orders","Data_Source") 

Catalog = dbutils.widgets.get("Catalog")
Data_Source = dbutils.widgets.get("Data_Source")
BasePath = "s3://sportsbar-us1/orders"
LandingPath = f"{BasePath}/landing/" 
ProcessedPath = f"{BasePath}/processed/"
print(BasePath)
print(LandingPath)
print(ProcessedPath) 

s3://sportsbar-us1/orders
s3://sportsbar-us1/orders/landing/
s3://sportsbar-us1/orders/processed/


In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(LandingPath)
        .withColumn("read_timestamp", current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
) 

# Print the toal rows: 
print("Total Rows:",df.count()) 
df.show()

No files found in s3://sportsbar-us1/orders/landing/
Skipping read operation.


In [0]:
df.write\
 .format("delta") \
 .option("mergeSchema", "true") \
 .mode("append") \
 .saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8074779971139234>, line 5
      1 df.write\
      2  .format("delta") \
      3  .option("mergeSchema", "true") \
      4  .mode("append") \
----> 5  .saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

File <command-8074779971139233>, line 2
      1 df = (
----> 2     spark.read.format("csv")
      3         .option("header", True)
      4         .option("inferSchema", True)
      5         .load(LandingPath)
      6         .withColumn("read_timestamp", current_timestamp())
      7         .select("*", "_metadata.file_name", "_metadata.file_size")
      8 ) 
     10 # Print the toal rows: 
     11 print("Total Rows:",df.count()) 

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for CSV. It must be specified manually. SQLSTATE: 42KD9

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.

### Moving files from landing to processed directory

In [0]:
files = dbutils.fs.ls(LandingPath)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{ProcessedPath}/{file_info.name}",
        True
    )

### Silver Transformation

In [0]:
df_orders = spark.read.table(f"{Catalog}.{bronze_schema}.{Data_Source}")
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891403|     NULL|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-20

In [0]:
# Keep only the rows where order_qty is present
df_orders = df_orders.filter(df_orders["order_qty"].isNotNull()) 
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|Tuesday, July 01,.

In [0]:
# Clean the customer_id keep only the numeric values other values to 0 and cast to string
df_orders = df_orders.withColumn("customer_id", when(df_orders["customer_id"].rlike("^[0-9]+$"), df_orders["customer_id"]).otherwise(99999).cast("string"))
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|Tuesday, July 01,...|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|Tuesday, July 01,.

In [0]:
# Replace the weekday name from the order placement day: 
df_orders = df_orders.withColumn("order_placement_date", regexp_replace(df_orders["order_placement_date"], r"^[A-Za-z]+,\s*", "")) 
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025/07/01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025/07/01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|       July 01, 2025|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|       July 01, 2025|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          01-07-2025|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|       July 01, 20

In [0]:
# Parasing order placement date from multiple formats: 
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy","MMMM dd, yyyy"] 
df_orders = df_orders.withColumn("order_placement_date", coalesce(*[try_to_date(df_orders["order_placement_date"], fmt) for fmt in date_formats]))
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025-07-01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|          2025-07-

In [0]:
# Drop duplicates: 
df_orders = df_orders.dropDuplicates(["customer_id","order_id","order_placement_date","order_qty","product_id"])
df_orders.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FJUL33320501|          2025-07-01|     789320|  25891203|    150.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891301|     46.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891201|    354.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33320501|          2025-07-01|     789320|  25891501|    249.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|          2025-07-01|     789401|  25891302|     40.0|2026-09-11 15:48:...|orders_2025_07_01...|    20744|
|FJUL33401603|          2025-07-

In [0]:
df_orders.printSchema()
# Converting product_id into string: 
df_orders = df_orders.withColumn("product_id", df_orders["product_id"].cast("string"))
df_orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: date (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



In [0]:
# Get the max amd min order placement date: 
df_orders.agg(min("order_placement_date").alias("min_date"), max("order_placement_date").alias("max_date")).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-07-01|2025-11-30|
+----------+----------+



In [0]:
# Join with the products table to get the product code: 
df_products = spark.read.table(f"{Catalog}.{silver_schema}.products")
df_joined = df_orders.join(df_products, df_orders["product_id"] == df_products["product_id"],'inner').select(df_orders["*"], df_products["product_code"])
df_joined.show(truncate=False)


+------------+--------------------+-----------+----------+---------+--------------------------+---------------------+---------+----------------------------------------------------------------+
|order_id    |order_placement_date|customer_id|product_id|order_qty|read_timestamp            |file_name            |file_size|product_code                                                    |
+------------+--------------------+-----------+----------+---------+--------------------------+---------------------+---------+----------------------------------------------------------------+
|FJUL32721603|2025-07-01          |99999      |25891202  |169.0    |2026-09-11 15:48:15.649157|orders_2025_07_01.csv|20744    |0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|
|FJUL33501401|2025-07-01          |789501     |25891303  |57.0     |2026-09-11 15:48:15.649157|orders_2025_07_01.csv|20744    |c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|
|FJUL34903603|2025-07-01          |

### Write it to the silver schema

In [0]:
df_joined.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{Catalog}.{silver_schema}.{Data_Source}")

### Gold Data Processing

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {Catalog}.{silver_schema}.{Data_Source}")
df_gold.show(2,truncate=False)

+------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|order_id    |date      |customer_code|product_code                                                    |product_id|sold_quantity|
+------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|FJUL32721603|2025-07-01|99999        |0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|25891202  |169.0        |
|FJUL33501401|2025-07-01|789501       |c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|25891303  |57.0         |
+------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
only showing top 2 rows


In [0]:
df_gold.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("overwriteSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{Catalog}.{gold_schema}.sb_fact_{Data_Source}")

### Merging with parent company: 


In [0]:
df_child = spark.sql(f"SELECT * FROM {Catalog}.{gold_schema}.sb_fact_{Data_Source}")
df_child.show(200,truncate=False)


+-------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|order_id     |date      |customer_code|product_code                                                    |product_id|sold_quantity|
+-------------+----------+-------------+----------------------------------------------------------------+----------+-------------+
|FJUL32721603 |2025-07-01|99999        |0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|25891202  |169.0        |
|FJUL33501401 |2025-07-01|789501       |c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|25891303  |57.0         |
|FJUL34903603 |2025-07-01|789903       |ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|25891501  |136.0        |
|FJUL32203402 |2025-07-01|789203       |c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|25891303  |64.0         |
|FJUL33421203 |2025-07-01|789421       |889c67757ece9c973791dfbc2d47b026a3342cc7255

In [0]:
# We want to make a monthly comparison but the child data is in daily format: 
df_monthly = df_child.withColumn("date", trunc("date", "month")).groupBy("date","product_code","customer_code").agg(sum("sold_quantity").alias("sold_quantity"))
df_monthly.show(200)

+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|0cb7b2f42657b625f...|        99999|      14539.0|
|2025-07-01|c68834ceaff15846b...|       789501|        676.0|
|2025-07-01|ee1f7df9cf660ef02...|       789903|       1924.0|
|2025-07-01|c68834ceaff15846b...|       789203|        365.0|
|2025-07-01|889c67757ece9c973...|       789421|       3318.0|
|2025-07-01|778c2a7aa27bfdb21...|       789102|       1685.0|
|2025-07-01|3cab59f0592428527...|       789103|        857.0|
|2025-07-01|0cb7b2f42657b625f...|       789401|       2103.0|
|2025-07-01|716fa4e54b7894c91...|       789520|       1771.0|
|2025-07-01|3cab59f0592428527...|       789202|       1038.0|
|2025-07-01|102628255d24304d6...|       789601|       5033.0|
|2025-07-01|da6bfc596c1360ca0...|        99999|      15323.0|
|2025-07-01|da6bfc596c1360ca0...|       789122|       3983.0|
|2025-07

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{Catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]